# V-SyncLock × AthletePose3D on Colab

Runs the 2D pose detector on AthletePose3D videos **inside Colab**, so the 35 GB archive never touches the laptop.
Only the extracted keypoints (MB-scale) are saved back to your Drive.

**Before running:** open the [AthletePose3D Drive folder](https://drive.google.com/drive/folders/10YnMJAluiscnLkrdiluIeehNetdry5Ft) →
right-click → *Organize → Add shortcut* → choose **My Drive**.
By using the data you accept the [license agreement](https://github.com/calvinyeungck/AthletePose3D/tree/main/license)
(non-commercial research; no redistribution or modified copies without written permission — keep outputs private).

Run **Step 1** first and send its printed output back; Steps 2–3 are adjusted to the real file layout afterwards.

## Step 0 — mount Drive and locate the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # remount so a newly added shortcut is visible

import os
BASE = '/content/drive/MyDrive'
# walk My Drive following shortcuts; stop descending below 4 levels to stay fast
hits = []
for root, dirs, files in os.walk(BASE, followlinks=True):
    if root[len(BASE):].count(os.sep) >= 4:
        dirs[:] = []
    if 'data.zip' in files and os.path.exists(os.path.join(root, 'cam_param.json')):
        hits.append(os.path.join(root, 'data.zip'))
print('data.zip candidates:', hits)
assert hits, ('data.zip not found under My Drive. Add the AthletePose3D folder as a shortcut '
              '(right-click > Organize > Add shortcut > My Drive), wait a minute, and re-run.')
DATA_ZIP = hits[0]
ROOT = os.path.dirname(DATA_ZIP)
print('dataset folder:', ROOT)
print(os.listdir(ROOT))

## Step 1 — inspect the archive without extracting it (send this output back)

In [ ]:
import zipfile, collections, json, re
zf = zipfile.ZipFile(DATA_ZIP)
infos = zf.infolist()
print(f'{len(infos)} entries, {sum(i.file_size for i in infos)/1e9:.1f} GB uncompressed')

# directory tree summary: count and size per top-2 path level and extension
agg = collections.defaultdict(lambda: [0, 0])
for i in infos:
    parts = i.filename.split('/')
    key = '/'.join(parts[:3]) + '  [' + os.path.splitext(i.filename)[1] + ']'
    agg[key][0] += 1; agg[key][1] += i.file_size
for k, (n, s) in sorted(agg.items()):
    print(f'{n:6d} files {s/1e9:7.2f} GB  {k}')

# a few full names per extension
byext = collections.defaultdict(list)
for i in infos:
    byext[os.path.splitext(i.filename)[1]].append(i.filename)
for e, names in byext.items():
    print('\n', e, len(names)); print('  ' + '\n  '.join(names[:8]))

In [ ]:
# metadata JSON of one running video and one non-running video, plus a .npy header
def show_json(name, limit=1500):
    with zf.open(name) as f:
        txt = f.read().decode('utf-8', 'replace')
    print('====', name); print(txt[:limit])

jsons = [n for n in byext.get('.json', [])]
run = [n for n in jsons if 'run' in n.lower()][:1]
other = [n for n in jsons if 'run' not in n.lower()][:1]
for n in run + other:
    show_json(n)

import numpy as np, io
npys = byext.get('.npy', [])
for n in (npys[:1] + [x for x in npys if 'run' in x.lower()][:1]):
    with zf.open(n) as f:
        a = np.load(io.BytesIO(f.read()), allow_pickle=True)
    print('====', n, type(a), getattr(a, 'shape', None), getattr(a, 'dtype', None))

In [ ]:
# calibration file and per-camera frame counts / fps of the running videos
cam = glob.glob(ROOT + '/cam_param.json')
if cam:
    P = json.load(open(cam[0]))
    print('cam_param keys:', list(P)[:20])
    k0 = list(P)[0]; print(k0, '->', json.dumps(P[k0])[:600])

import cv2, tempfile
mp4 = [n for n in byext.get('.mp4', []) if 'run' in n.lower()]
print(len(mp4), 'running videos')
groups = collections.defaultdict(list)
for n in mp4:
    groups[re.sub(r'_cam_?\d+', '', n)].append(n)
for g, names in list(groups.items())[:2]:            # two trials, all cameras
    for n in sorted(names):
        tmp = '/content/_probe.mp4'
        with zf.open(n) as src, open(tmp, 'wb') as dst:
            dst.write(src.read())
        c = cv2.VideoCapture(tmp)
        print(f'{n}: {int(c.get(7))} frames, {c.get(5):.3f} fps, {int(c.get(3))}x{int(c.get(4))}')
        c.release()

## Step 2 — extract running videos to Colab's local disk
(Adjusted after Step 1. By default: every `.mp4` and `.json` whose path contains `run`.)

In [ ]:
LOCAL = '/content/ap3d'
os.makedirs(LOCAL, exist_ok=True)
sel = [i for i in infos if 'run' in i.filename.lower() and i.filename.endswith(('.mp4', '.json'))]
print(len(sel), 'files', sum(i.file_size for i in sel)/1e9, 'GB')
for i in sel:
    zf.extract(i, LOCAL)
print('done')

## Step 3 — MediaPipe Pose → COCO-19 keypoints (same mapping as `vsynclock/realworld/pose2d.py`)
Every frame is processed (120 fps), so any decimation phase can be built later. Output: one `.npz` per video in
`MyDrive/ap3d_keypoints/` (private — do not share).

In [ ]:
!pip -q install mediapipe==0.10.30   # Colab is on Python 3.13; >=0.10.30 only ships the Tasks API
!wget -q -O /content/pose_landmarker_full.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task

In [ ]:
import numpy as np, cv2, os, time, glob
import mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
from concurrent.futures import ProcessPoolExecutor

MODEL = '/content/pose_landmarker_full.task'               # BlazePose full (= legacy model_complexity=1)
OUT_DIR = '/content/drive/MyDrive/ap3d_keypoints'      # private: license forbids redistribution
_MID = {0: (11, 12), 2: (23, 24)}
_DIRECT = {1: 0, 3: 11, 4: 13, 5: 15, 6: 23, 7: 25, 8: 27, 9: 12, 10: 14, 11: 16,
           12: 24, 13: 26, 14: 28, 15: 2, 16: 7, 17: 5, 18: 8}

def to_coco19(lm):
    out = np.full((19, 2), np.nan)
    for j, s in _DIRECT.items(): out[j] = lm[s]
    for j, (a, b) in _MID.items(): out[j] = 0.5 * (lm[a] + lm[b])
    return out

def run_video(path, min_vis=0.3):
    os.makedirs(OUT_DIR, exist_ok=True)
    split, subj = path.split('/')[-3], path.split('/')[-2]     # names repeat across subjects
    out = os.path.join(OUT_DIR, f'{split}_{subj}_' + os.path.basename(path).replace('.mp4', '.npz'))
    if os.path.exists(out):
        return out, 'skip'
    opts = vision.PoseLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=MODEL),
        running_mode=vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5)
    cap = cv2.VideoCapture(path); W, H, fps = cap.get(3), cap.get(4), cap.get(5) or 120.0
    xy, vis, i = [], [], 0
    with vision.PoseLandmarker.create_from_options(opts) as det:
        while True:
            ok, bgr = cap.read()
            if not ok: break
            img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            res = det.detect_for_video(img, int(round(i * 1000.0 / fps)))   # strictly increasing ms
            lm = np.full((33, 2), np.nan); v = np.zeros(33)
            if res.pose_landmarks:
                for k, p in enumerate(res.pose_landmarks[0]):
                    v[k] = p.visibility
                    if p.visibility >= min_vis: lm[k] = (p.x * W, p.y * H)
            xy.append(to_coco19(lm)); vis.append(v); i += 1
    cap.release()
    np.savez(out, xy=np.asarray(xy, np.float32), vis=np.asarray(vis, np.float32), fps=fps, src=path)
    return out, f'{i} frames'

videos = sorted(glob.glob(LOCAL + '/**/*.mp4', recursive=True))
SELECT = videos[:1]          # <- set to the chosen subset after timing one video
print(len(videos), 'videos available;', len(SELECT), 'selected')
t0 = time.time()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for out, msg in ex.map(run_video, SELECT):
        print(f'{time.time()-t0:7.0f}s  {os.path.basename(out)}  {msg}')


In [ ]:
# copy the per-video JSON metadata and calibration next to the keypoints (still private)
import shutil
dst = '/content/drive/MyDrive/ap3d_keypoints'
for j in glob.glob(LOCAL + '/**/*.json', recursive=True):
    shutil.copy(j, dst)
for c in glob.glob(ROOT + '/cam_param.json'):
    shutil.copy(c, dst)
print(len(os.listdir(dst)), 'files in', dst)